# 01 — KIỂM ĐỊNH DATA LEAKAGE
# Giai đoạn 2

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from common import training

OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# 1. Đọc dữ liệu Window (Thay vì file feature 40 dòng cũ)
print("Đang tải dữ liệu window...")
windows_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_raw.parquet")
manifest = pd.read_csv("../giai_doan_1_tien_xu_ly/outputs/tables/manifest_filtered.csv")

# 2. Khôi phục file_id và ghép load_hp
manifest['file_id'] = manifest.apply(
    lambda row: f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils', '')}_{row['file_path']}", axis=1
)
windows_df = windows_df.merge(manifest[['file_id', 'load_hp']], on='file_id', how='left')
windows_df = windows_df.dropna(subset=['load_hp']).reset_index(drop=True)
windows_df['load_hp'] = windows_df['load_hp'].astype(int)

# 3. Hàm trích xuất đặc trưng THỜI GIAN THỰC cho 1 window
def extract_window_features(window_data):
    w = np.array(window_data)
    rms = np.sqrt(np.mean(w**2))
    return {
        'time_mean': np.mean(w),
        'time_std': np.std(w),
        'time_rms': rms,
        'time_peak': np.max(np.abs(w)),
        'time_skew': skew(w),
        'time_kurtosis': kurtosis(w),
        'time_crest': np.max(np.abs(w)) / (rms + 1e-8)
    }

# Áp dụng tính toán (mất khoảng vài giây)
print("Đang tính toán đặc trưng cho từng window...")
features_list = windows_df['window'].apply(extract_window_features).tolist()
feature_df = pd.DataFrame(features_list)

# Ghép nhãn và siêu dữ liệu
feature_df['label'] = windows_df['label']
feature_df['file_id'] = windows_df['file_id']
feature_df['load_hp'] = windows_df['load_hp']

feature_cols = [col for col in feature_df.columns if col.startswith('time_')]

print(f"Hoàn tất! Tổng số mẫu (windows): {len(feature_df)}")
print(f"Số lượng đặc trưng: {len(feature_cols)}")

X = feature_df[feature_cols].values
y = feature_df['label'].values
loads = feature_df['load_hp'].values

Đang tải dữ liệu window...
Đang tính toán đặc trưng cho từng window...
Hoàn tất! Tổng số mẫu (windows): 5886
Số lượng đặc trưng: 7


In [6]:
def rf_factory():
    return RandomForestClassifier(n_estimators=100, random_state=42)

# Chạy đánh giá LOLO (Train trên tải A,B,C -> Test trên tải D)
per_fold, summary = training.run_lolo_evaluation(
    feature_df,
    feature_cols=feature_cols,
    estimator_factory=rf_factory,
    label_col='label',
    load_col='load_hp',
    val_ratio=0.2,
    seed=42,
    loads=(0, 1, 2, 3),
    use_val_for_fit=True  
)

le = LabelEncoder()
y_enc = le.fit_transform(y)

mean_acc_lolo = summary['accuracy_mean']
std_acc_lolo = summary['accuracy_std']
print(f"\nFile-based + LOLO: Accuracy = {mean_acc_lolo:.4f} ± {std_acc_lolo:.4f}")


File-based + LOLO: Accuracy = 0.9633 ± 0.0490


In [4]:
print("\n--- THỰC THI RQ1: RANDOM SPLIT vs FILE-BASED SPLIT ---")

# 1. Random Window Split (Chia ngẫu nhiên từng mảnh window nhỏ - Gây rò rỉ dữ liệu)
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    feature_df[feature_cols], feature_df['label'], test_size=0.2, random_state=42
)
rf_rnd = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rnd.fit(X_train_rnd, y_train_rnd)
acc_rnd = accuracy_score(y_test_rnd, rf_rnd.predict(X_test_rnd))

# 2. File-Based Split (Chia theo nguyên cụm file_id - Ngăn chặn rò rỉ dữ liệu)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(feature_df[feature_cols], feature_df['label'], groups=feature_df['file_id']))

X_train_file = feature_df.loc[train_idx, feature_cols]
y_train_file = feature_df.loc[train_idx, 'label']
X_test_file = feature_df.loc[test_idx, feature_cols]
y_test_file = feature_df.loc[test_idx, 'label']

rf_file = RandomForestClassifier(n_estimators=100, random_state=42)
rf_file.fit(X_train_file, y_train_file)
acc_file = accuracy_score(y_test_file, rf_file.predict(X_test_file))

# 3. Kết xuất bảng báo cáo
delta_acc = acc_rnd - acc_file

print(f"Accuracy (Random Split - Bị Leakage): {acc_rnd:.4f}")
print(f"Accuracy (File-based Split - Thực tế): {acc_file:.4f}")
print(f"Δ Accuracy (Độ chênh lệch): {delta_acc:.4f}")

# Lưu kết quả
results_rq1 = pd.DataFrame({
    'Method': ['Random Split (Leakage)', 'File-based Split (Thực tế)'],
    'Accuracy': [acc_rnd, acc_file]
})
results_rq1.to_csv(TABLES_DIR / "rq1_random_vs_filebased.csv", index=False)


--- THỰC THI RQ1: RANDOM SPLIT vs FILE-BASED SPLIT ---
Accuracy (Random Split - Bị Leakage): 0.9966
Accuracy (File-based Split - Thực tế): 0.8937
Δ Accuracy (Độ chênh lệch): 0.1029
